## 先加入統計檢定工具函數

In [2]:
import numpy as np
import pandas as pd
from scipy import stats


def summarize_return_significance(
    returns: pd.Series,
    alternative: str = "greater",
) -> pd.Series:
    """
    Test whether average return is statistically different from 0.

    alternative:
    - "greater": mean return > 0
    - "two-sided": mean return != 0
    """

    r = returns.dropna()

    if len(r) < 2:
        return pd.Series({
            "count": len(r),
            "mean": np.nan,
            "median": np.nan,
            "std": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan,
            "win_rate": np.nan,
            "sharpe_like": np.nan,
        })

    t_stat, p_two_sided = stats.ttest_1samp(r, 0)

    if alternative == "greater":
        if t_stat > 0:
            p_value = p_two_sided / 2
        else:
            p_value = 1 - p_two_sided / 2
    else:
        p_value = p_two_sided

    return pd.Series({
        "count": len(r),
        "mean": r.mean(),
        "median": r.median(),
        "std": r.std(),
        "t_stat": t_stat,
        "p_value": p_value,
        "win_rate": (r > 0).mean(),
        "sharpe_like": r.mean() / r.std() if r.std() != 0 else np.nan,
    })

## 建立不同持有期比較

In [2]:
holding_windows = [
    (-5, -1),
    (-10, -1),
    (-20, -1),
    (-30, -1),
    (-50, -1),
]

window_results = []

for entry_day, exit_day in holding_windows:
    trades = calculate_event_trade_return_enriched(
        event_data=event_data,
        price_col="adj_close",
        entry_day=entry_day,
        exit_day=exit_day,
        fee_rate=0.0008,
    )

    summary = summarize_event_trades(trades)
    significance = summarize_return_significance(
        trades["net_return"],
        alternative="greater",
    )

    row = {
        "entry_day": entry_day,
        "exit_day": exit_day,
        "holding_days": exit_day - entry_day,
        "avg_daily_net_return": trades["net_return"].mean() / (exit_day - entry_day),
        **summary.add_prefix("raw_").to_dict(),
        **significance.add_prefix("sig_").to_dict(),
    }

    window_results.append(row)

window_comparison = pd.DataFrame(window_results)

window_comparison.sort_values(
    ["sig_t_stat", "avg_daily_net_return"],
    ascending=[False, False],
)

NameError: name 'calculate_event_trade_return_enriched' is not defined

## 加入 benchmark-adjusted return

In [ ]:
def add_benchmark_window_return(
    trades: pd.DataFrame,
    panel_all: pd.DataFrame,
    benchmark_symbol: str = "0050",
    price_col: str = "adj_close",
) -> pd.DataFrame:
    """
    Add benchmark return over the same entry-exit window.
    """

    benchmark = (
        panel_all[panel_all["symbol"] == benchmark_symbol]
        .sort_values("date")
        [["date", price_col]]
        .copy()
    )

    benchmark = benchmark.rename(
        columns={
            price_col: "benchmark_price",
        }
    )

    df = trades.copy()

    entry_benchmark = benchmark.rename(
        columns={
            "date": "entry_date",
            "benchmark_price": "benchmark_entry_price",
        }
    )

    exit_benchmark = benchmark.rename(
        columns={
            "date": "exit_date",
            "benchmark_price": "benchmark_exit_price",
        }
    )

    df = df.merge(entry_benchmark, on="entry_date", how="left")
    df = df.merge(exit_benchmark, on="exit_date", how="left")

    df["benchmark_return"] = (
        df["benchmark_exit_price"]
        / df["benchmark_entry_price"]
        - 1
    )

    df["excess_return_vs_benchmark"] = (
        df["net_return"]
        - df["benchmark_return"]
    )

    return df

In [ ]:
benchmark_adjusted_results = []

for entry_day, exit_day in holding_windows:
    trades = calculate_event_trade_return_enriched(
        event_data=event_data,
        price_col="adj_close",
        entry_day=entry_day,
        exit_day=exit_day,
        fee_rate=0.0008,
    )

    trades = add_benchmark_window_return(
        trades=trades,
        panel_all=panel_all,
        benchmark_symbol="0050",
        price_col="adj_close",
    )

    sig_excess = summarize_return_significance(
        trades["excess_return_vs_benchmark"],
        alternative="greater",
    )

    row = {
        "entry_day": entry_day,
        "exit_day": exit_day,
        "holding_days": exit_day - entry_day,
        "avg_net_return": trades["net_return"].mean(),
        "avg_benchmark_return": trades["benchmark_return"].mean(),
        "avg_excess_return": trades["excess_return_vs_benchmark"].mean(),
        "avg_daily_excess_return": (
            trades["excess_return_vs_benchmark"].mean()
            / (exit_day - entry_day)
        ),
        **sig_excess.add_prefix("excess_sig_").to_dict(),
    }

    benchmark_adjusted_results.append(row)

benchmark_adjusted_comparison = pd.DataFrame(benchmark_adjusted_results)

benchmark_adjusted_comparison.sort_values(
    ["excess_sig_t_stat", "avg_daily_excess_return"],
    ascending=[False, False],
)

## 做「分段報酬」證明 alpha 集中在哪裡

In [ ]:
segment_windows = [
    (-50, -31),
    (-30, -21),
    (-20, -11),
    (-10, -1),
]

segment_results = []

for entry_day, exit_day in segment_windows:
    trades = calculate_event_trade_return_enriched(
        event_data=event_data,
        price_col="adj_close",
        entry_day=entry_day,
        exit_day=exit_day,
        fee_rate=0.0008,
    )

    trades = add_benchmark_window_return(
        trades=trades,
        panel_all=panel_all,
        benchmark_symbol="0050",
        price_col="adj_close",
    )

    sig_raw = summarize_return_significance(
        trades["net_return"],
        alternative="greater",
    )

    sig_excess = summarize_return_significance(
        trades["excess_return_vs_benchmark"],
        alternative="greater",
    )

    row = {
        "segment": f"{entry_day} to {exit_day}",
        "entry_day": entry_day,
        "exit_day": exit_day,
        "holding_days": exit_day - entry_day,
        "avg_net_return": trades["net_return"].mean(),
        "avg_daily_net_return": trades["net_return"].mean() / (exit_day - entry_day),
        "avg_excess_return": trades["excess_return_vs_benchmark"].mean(),
        "avg_daily_excess_return": (
            trades["excess_return_vs_benchmark"].mean()
            / (exit_day - entry_day)
        ),
        "raw_t_stat": sig_raw["t_stat"],
        "raw_p_value": sig_raw["p_value"],
        "excess_t_stat": sig_excess["t_stat"],
        "excess_p_value": sig_excess["p_value"],
        "win_rate": (trades["net_return"] > 0).mean(),
    }

    segment_results.append(row)

segment_comparison = pd.DataFrame(segment_results)

segment_comparison

## Placebo Test：證明不是隨便買也會賺

In [ ]:
def generate_random_windows(
    panel: pd.DataFrame,
    events: pd.DataFrame,
    symbol: str,
    holding_days: int,
    n_samples: int = 1000,
    buffer_days: int = 20,
    price_col: str = "adj_close",
    random_state: int = 42,
) -> pd.Series:
    """
    Generate random non-event window returns for one symbol.
    """

    rng = np.random.default_rng(random_state)

    p = (
        panel[panel["symbol"] == symbol]
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    event_dates = set(
        pd.to_datetime(events.loc[events["symbol"] == symbol, "ex_date"])
    )

    valid_returns = []

    max_start = len(p) - holding_days - 1

    if max_start <= 0:
        return pd.Series(dtype=float)

    for _ in range(n_samples):
        start_idx = rng.integers(0, max_start)
        end_idx = start_idx + holding_days

        start_date = p.loc[start_idx, "date"]
        end_date = p.loc[end_idx, "date"]

        # 排除靠近除息日的窗口，避免 placebo 被事件污染
        is_near_event = False

        for ex_date in event_dates:
            if abs((start_date - ex_date).days) <= buffer_days:
                is_near_event = True
                break

        if is_near_event:
            continue

        start_price = p.loc[start_idx, price_col]
        end_price = p.loc[end_idx, price_col]

        if pd.notna(start_price) and pd.notna(end_price) and start_price != 0:
            valid_returns.append(end_price / start_price - 1)

    return pd.Series(valid_returns)

In [ ]:
actual_trades = calculate_event_trade_return_enriched(
    event_data=event_data,
    price_col="adj_close",
    entry_day=-10,
    exit_day=-1,
    fee_rate=0.0008,
)

actual_mean = actual_trades["net_return"].mean()

random_returns_all = []

for symbol in actual_trades["symbol"].unique():
    random_returns = generate_random_windows(
        panel=panel_all,
        events=event_summary,
        symbol=symbol,
        holding_days=9,
        n_samples=3000,
        buffer_days=30,
        price_col="adj_close",
        random_state=42,
    )

    random_returns_all.append(random_returns)

random_returns_all = pd.concat(random_returns_all, ignore_index=True)

placebo_p_value = (
    random_returns_all.mean()
    >= actual_mean
)

placebo_summary = pd.Series({
    "actual_mean_return": actual_mean,
    "random_mean_return": random_returns_all.mean(),
    "random_median_return": random_returns_all.median(),
    "actual_minus_random": actual_mean - random_returns_all.mean(),
    "random_sample_count": len(random_returns_all),
})

placebo_summary

In [ ]:
def bootstrap_random_mean_pvalue(
    actual_mean: float,
    random_returns: pd.Series,
    event_count: int,
    n_bootstrap: int = 10000,
    random_state: int = 42,
) -> pd.Series:
    """
    Bootstrap random window mean returns and compare with actual event mean.
    """

    rng = np.random.default_rng(random_state)

    r = random_returns.dropna().to_numpy()

    boot_means = []

    for _ in range(n_bootstrap):
        sample = rng.choice(r, size=event_count, replace=True)
        boot_means.append(sample.mean())

    boot_means = np.array(boot_means)

    p_value = (boot_means >= actual_mean).mean()

    return pd.Series({
        "actual_mean": actual_mean,
        "bootstrap_random_mean": boot_means.mean(),
        "bootstrap_random_std": boot_means.std(),
        "p_value_actual_greater_than_random": p_value,
        "random_mean_95pct": np.quantile(boot_means, 0.95),
        "random_mean_99pct": np.quantile(boot_means, 0.99),
    })


placebo_test_result = bootstrap_random_mean_pvalue(
    actual_mean=actual_mean,
    random_returns=random_returns_all,
    event_count=len(actual_trades),
    n_bootstrap=10000,
    random_state=42,
)

placebo_test_result